In [1]:
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
import re
import json

def printShape(df, cols=[], msg=''):
    
    print(df.shape, end='  ')
    for col in cols:
        if col in df.columns:
            print(col, df[col].nunique(), end='  ')
    print(msg, flush=True)
    
    return df

PROJDATA = '../data'
PUBMED = f'{PROJDATA}/pubmed'
PLOS = f'{PROJDATA}/plos'
LARGE = f'{PROJDATA}/large_files'
EXTERNAL = '/scratch/fl1092/pubmed'
EXTERNAL2 = '/scratch/fl1092/LLM_use_RAD/data/'

# Classification

In [2]:
%%time
classification = (
    pd.read_csv(
        f'{EXTERNAL2}/ClassificationsAggregated.csv',
        usecols=['File','DetectAbsScore', 'DetectAbs', 'DetectAbs7','DetectIntro7', 'BinoAbs']
    )
    .pipe(printShape, cols=['File'])
    
    .drop_duplicates(subset=['File','BinoAbs','DetectAbs','DetectIntro7'])
    .pipe(printShape, cols=['File',])
    
    .drop_duplicates(subset=['File'], keep=False)
    .pipe(printShape, cols=['File']) # (6865636, 10)  File 6865636 
)

(6865733, 6)  File 6865636  
(6865636, 6)  File 6865636  
(6865636, 6)  File 6865636  
CPU times: user 40.4 s, sys: 965 ms, total: 41.4 s
Wall time: 41.5 s


In [3]:
classification.to_csv(f'{LARGE}/ClassificationsAggregated.csv', index=False)

# Relative acceptance delay (RAD)

## Paper journal

In [4]:
def extract_letters_if_letters_then_digits(s: str):
    """
    If the string starts with letters only, followed by digits
    that may include parentheses, return the letter part.
    Otherwise, return None.
    """
    match = re.fullmatch(r'([A-Za-z]+)[\d()]+', s)
    if match:
        return match.group(1)
    return None


def getJournalCode(x):
    
    x = x.split('/')
    
    if len(x) >2:
        return x[1]
    
    x = x[1]

    if '.' in x:
        
        x = x.split('.')
        
        if len(x) == 2:
            return x[0]
        elif len(x) == 3:
            return '.'.join(x[:2])
        else:
            return '.'.join(x[:3])
    
    elif '-' in x:
        x = x.split('-')
        
        if len(x) == 2:
            return x[0]
        else:
            return '-'.join(x[:2])
    
    else:
        return extract_letters_if_letters_then_digits(x)

In [5]:
%%time
paperJournal = (
    pd.concat([
        (
            pd.read_csv(file, usecols=['File','doi'], dtype={'doi':str})
        )
        for file in tqdm(glob.glob(f'{PUBMED}/id/*.csv'))
    ], ignore_index=True, sort=False)
    .pipe(printShape, cols=['File'])

    ### process paper publisher and journal using paper DOI
    .dropna(subset=['doi'])
    .assign(Publisher=lambda df: df.doi.apply(lambda x: x.split('/')[0]))
    .pipe(printShape, cols=['File'])
    
    .assign(Journal=lambda df: df.doi.apply(getJournalCode))
    .assign(Journal=lambda df: df.apply(lambda row: row['Publisher'] if pd.isna(row['Journal']) else row['Journal'], axis=1))
)

100%|██████████| 42/42 [00:10<00:00,  3.99it/s]


(7520282, 2)  File 7472079  
(7095216, 3)  File 7047629  
CPU times: user 1min 29s, sys: 1.26 s, total: 1min 30s
Wall time: 1min 31s


In [6]:
paperJournal.head(2)

,doi,File,Publisher,Journal
0,10.3390/ani11030823,PMC008xxxxxx/PMC8000001.xml,10.3390,ani
1,10.3390/ma14061382,PMC008xxxxxx/PMC8000002.xml,10.3390,ma


In [7]:
paperJournal.to_csv(f'{LARGE}/PaperJournal.csv', index=False)

## Paper year

In [8]:
%%time
paperYear = (
    pd.read_csv(
        f'{PROJDATA}/large_files/AcceptanceDelay.csv',
        usecols=['File','Accepted','AcptYear'], parse_dates=['Accepted']
    )
    .dropna(subset=['Accepted']).pipe(printShape, cols=['File']) # 6060809
    .assign(AcptYear=lambda df: df.AcptYear.astype(int))
)

(6060809, 3)  File 6060809  
CPU times: user 11 s, sys: 455 ms, total: 11.4 s
Wall time: 11.5 s


In [9]:
paperYear.to_csv(f'{LARGE}/PaperYear.csv', index=False)

## RAD

In [10]:
%%time
paperJournal = pd.read_csv(f'{LARGE}/PaperJournal.csv', dtype={'Publisher':str}, keep_default_na=False)

classification=(
    pd.read_csv(f'{LARGE}/ClassificationsAggregated.csv', usecols=['File','DetectAbs7'])
    .dropna()[['File']]
)

delay = (
    pd.read_csv(f'{PROJDATA}/large_files/AcceptanceDelay.csv', usecols=['File','AcptDelay','AcptYear'])
    .dropna(subset=['AcptDelay']).pipe(printShape) # 5870180
)

(5870180, 3)  
CPU times: user 21.1 s, sys: 1.05 s, total: 22.2 s
Wall time: 22.3 s


In [11]:
%%time
RAD = (
    delay
    .merge(paperJournal, on='File')
    .merge(classification, on='File')
    .pipe(printShape, cols=['File']) # (5418115, 6)  File 5396278  
    
    .assign(
        RAD=lambda df:
        df.groupby(['AcptYear', 'Publisher', 'Journal'])['AcptDelay']
        .transform(lambda x: (x - x.mean()) )
    )
    
    .assign(
        RADzscore=lambda df:
        df.groupby(['AcptYear', 'Publisher', 'Journal'])['AcptDelay']
        .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
    )

    .dropna(subset=['RADzscore'])
    .drop_duplicates(subset=['File','Publisher','Journal','AcptDelay','AcptYear'])
    .pipe(printShape, cols=['File']) # (5354166, 13)  File 5354121 
    
    .assign(AcptYear=lambda df: df.AcptYear.astype(int))

    .drop_duplicates(subset=['File','RAD'])
    .pipe(printShape, cols=['File'])

    .drop_duplicates(subset=['File'], keep=False)
    .pipe(printShape, cols=['File']) # 5354117
)

(5418074, 6)  File 5396278  
(5354166, 8)  File 5354121  
(5354125, 8)  File 5354121  
(5354117, 8)  File 5354117  
CPU times: user 1min 42s, sys: 1.53 s, total: 1min 44s
Wall time: 1min 44s


In [12]:
RAD.to_csv(
    f'{LARGE}/RAD.csv', index=False,
    columns=['File','Publisher','Journal','AcptDelay','AcptYear','RAD']
)

# Paper region

In [13]:
def processNorthSouth(row):

    # True: global south
    # False: global north

    r = row['region']
    sub = row['sub-region']
    country = row['country']

    if country == 'JP' or country == 'IL' or country == "KR":
        # Japan, Israel, and Korea are global north
        return False

    if r == 'Asia' or r == 'Africa':
        return True
    elif r == 'Americas':
        if sub == 'Latin America and the Caribbean':
            return True
        elif sub == 'Northern America':
            return False
        else:
            raise Error('Unknown subregion in Americas')
    elif r == 'Oceania':
        if sub == 'Australia and New Zealand':
            return False
        else:
            return True
    elif r == "Europe":
        return False

In [14]:
continents = (
    pd.read_csv(f'{PROJDATA}/continents2.csv', usecols=['alpha-2', 'region', 'sub-region', 'name'])
    .rename(columns={'alpha-2': 'country'})
    .pipe(printShape)
    
    .assign(GlobalSouth=lambda df: df.apply(processNorthSouth, axis=1))
    
    .assign(name=lambda df: df.name.apply(lambda x: x.lower()))
)

(249, 4)  


In [15]:
continents.head()

,name,country,region,sub-region,GlobalSouth
0,afghanistan,AF,Asia,Southern Asia,True
1,åland islands,AX,Europe,Northern Europe,False
2,albania,AL,Europe,Southern Europe,False
3,algeria,DZ,Africa,Northern Africa,True
4,american samoa,AS,Oceania,Polynesia,True


In [17]:
%%time
papAff = (
    pd.concat([
        pd.read_csv(file, usecols=['File','AuthorOrder','Affiliations'])
        for file in tqdm(glob.glob(f'{EXTERNAL}/author_aff/*.csv'))
    ], ignore_index=True, sort=False)
    .pipe(printShape, cols=['File'])
    
    .dropna(subset=['Affiliations'])
    .assign(AffCount=lambda df: df.Affiliations.apply(lambda x: len(x.split(';'))) )
    .pipe(printShape, cols=['File'])
)

100%|██████████| 42/42 [02:07<00:00,  3.04s/it]


(49854866, 3)  File 7233758  
(44085350, 4)  File 6243003  
CPU times: user 2min 57s, sys: 6.48 s, total: 3min 3s
Wall time: 3min 4s


In [18]:
%%time
singleAff = papAff.query('AffCount == 1').pipe(printShape, cols=['File'])
multiAff = papAff.query('AffCount > 1').pipe(printShape, cols=['File'])

(39468065, 4)  File 5748522  
(4617285, 4)  File 744791  
CPU times: user 9.76 s, sys: 317 ms, total: 10.1 s
Wall time: 10.1 s


In [19]:
%%time
expandDf = []

for ind, row in tqdm(multiAff.iterrows()):
    
    for aff in row['Affiliations'].split(';'):
    
        expandDf.append({
            'AuthorOrder': row['AuthorOrder'],
            'File': row['File'],
            'Affiliations': aff,
        })
        
expandDf = pd.DataFrame(expandDf)

4617285it [05:18, 14508.22it/s]


CPU times: user 5min 28s, sys: 3.68 s, total: 5min 32s
Wall time: 5min 34s


In [20]:
%%time
us_state_codes = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

us_states = [
    "alabama", "alaska", "arizona", "arkansas", "california",
    "colorado", "connecticut", "delaware", "florida", "georgia",
    "hawaii", "idaho", "illinois", "indiana", "iowa",
    "kansas", "kentucky", "louisiana", "maine", "maryland",
    "massachusetts", "michigan", "minnesota", "mississippi", "missouri",
    "montana", "nebraska", "nevada", "new hampshire", "new jersey",
    "new mexico", "new york", "north carolina", "north dakota", "ohio",
    "oklahoma", "oregon", "pennsylvania", "rhode island", "south carolina",
    "south dakota", "tennessee", "texas", "utah", "vermont",
    "virginia", "washington", "west virginia", "wisconsin", "wyoming"
]

canadian_province_codes = [
    "AB", "BC", "MB", "NB", "NL", "NS", "NT",
    "NU", "ON", "PE", "QC", "SK", "YT"
]

countryMap = {
    'united\nstates': 'united states',
    'usa': 'united states',
    'united states of america': 'united states',
    'u.s.a': 'united states',

    'u.k': 'united kingdom',
    'uk': 'united kingdom',
    'the netherlands': 'netherlands',
    'türkiye': 'turkey',

    'korea': 'south korea',
    'korea': 'south korea',
    'republic of korea': 'south korea',

    "people's republic of china": 'china',
    'people’s republic of china': 'china',
    'p.r. china': 'china',
    'pr china': 'china',
    'p. r. china': 'china',

    'brasil': 'brazil',
    'méxico':'mexico',
    'deutschland': 'germany',
    'españa': 'spain',
    
    'kingdom of saudi arabia': 'saudi arabia',
    'uae':'united arab emirates',
    'viet nam': 'vietnam',
    'russian federation': 'russia'
    
}

for state in us_state_codes:
    countryMap[state.lower()] = 'united states'
    countryMap[f'{state.lower()} usa'] = 'united states'

for state in us_states:
    countryMap[state.lower()] = 'united states'

for state in canadian_province_codes:
    countryMap[state.lower()] = 'canada'
    countryMap[f'{state.lower()} canada'] = 'canada'
    
countryNames = set(continents.name.values)
    
import re

def remove_emails_and_initials(text: str) -> str:
    pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b\s*(?:\([a-z.-]+\))?'
    return re.sub(pattern, '', text).strip()

def returnCountryListOfStrings(x):
    
    if len(x) >=1:
        candidate = x[-1]
        res = returnCountry(candidate)
        if res is not None:
            return res
        
    if len(x) >=2:
        candidate = ' '.join(x[-2:])
        res = returnCountry(candidate)
        if res is not None:
            return res
        
    if len(x) >=3:
        candidate = ' '.join(x[-3:])
        res = returnCountry(candidate)
        if res is not None:
            return res
        
    return None
        

def returnCountry(x):
    
    if x in countryNames:
        return x

    if x in countryMap:
        return countryMap[x]

    return None
    
        
def cleanCountry(x):
    
    x = remove_emails_and_initials(x.split(',')[-1].lower().strip('. \n'))
    
    res = returnCountry(x)
    if res is not None:
        return res
    
    xsplit = x.split()
    res = returnCountryListOfStrings(xsplit)
    if res is not None:
        return res

    if len(xsplit) >= 2 and xsplit[-1].isnumeric():
        
        res = returnCountry(xsplit[-2])
        if res is not None:
            return res

    return x
    

papAff = (
    pd.concat([singleAff.drop('AffCount', axis=1), expandDf], ignore_index=True, sort=False)
    .pipe(printShape, cols=['File'])
    
    .assign(Country=lambda df: df.Affiliations.apply(cleanCountry))
)

(58158217, 3)  File 6243003  
CPU times: user 2min 46s, sys: 788 ms, total: 2min 46s
Wall time: 2min 47s


In [21]:
%%time
papAffCountry = (
    papAff.rename(columns={'Country': 'name'})
    .merge(continents, on='name').pipe(printShape, cols=['File']) # 43675520, 6071751
)

(43675520, 8)  File 6071751  
CPU times: user 25.8 s, sys: 1.19 s, total: 27 s
Wall time: 27.1 s


In [22]:
%%time
paperAuthorCount = (
    papAff.groupby('File').AuthorOrder.nunique().reset_index()
    .rename(columns={'AuthorOrder':'Total'})
)

paperCountryCount = (
    papAffCountry.groupby(['File', 'country'])
    .AuthorOrder.nunique().reset_index()
    .rename(columns={'AuthorOrder':'AuthorCount'})
)

paperCountry = (
    paperAuthorCount.merge(paperCountryCount, on='File')
    .assign(Percentage=lambda df: df.AuthorCount/df.Total)
)

CPU times: user 53.1 s, sys: 1.42 s, total: 54.5 s
Wall time: 54.6 s


In [23]:
papMainCountry = (
    paperCountry.sort_values(by=['Percentage'], ascending=False)
    .drop_duplicates(subset=['File'], keep='first')
    .pipe(printShape, cols=['File']) # 6071579
)

(6071579, 5)  File 6071579  


In [24]:
papMainCountry.head()

,File,Total,country,AuthorCount,Percentage
8783129,PMC012xxxxxx/PMC12747802.xml,9,CN,9,1.0
4335818,PMC008xxxxxx/PMC8299608.xml,12,CN,12,1.0
4335814,PMC008xxxxxx/PMC8299605.xml,11,CN,11,1.0
4335811,PMC008xxxxxx/PMC8299601.xml,3,PL,3,1.0
4335810,PMC008xxxxxx/PMC8299600.xml,16,CN,16,1.0


In [25]:
papMainCountry.to_csv(
    f'{LARGE}/PaperMainCountry.csv', columns=['File', 'country', 'Percentage'], index=False
)